# Cadence - MP4 Random Sample Preprocessing Pipeline

## 1. Imports and configuration

In [1]:
from pathlib import Path
import csv
import json
import random
import re
import shutil
import subprocess
from typing import Any

SAMPLE_COUNT = 10
RANDOM_SEED = 42
MIN_WINDOW_SECONDS = 10 * 60  # 10 minutes
MAX_WINDOW_SECONDS = 15 * 60  # 15 minutes

WAV_SAMPLE_RATE = 16_000
MP3_SAMPLE_RATE = 44_100
MP3_BITRATE = "128k"

RECORDINGS_DIR = Path("/content/drive/MyDrive/Class Files/A2 CS 2025/Recordings")
LOCAL_OUTPUT_DIR = Path("/content/cadence_dataset_samples")
LOCAL_WAV_DIR = LOCAL_OUTPUT_DIR / "wav"
LOCAL_MP3_DIR = LOCAL_OUTPUT_DIR / "mp3"
LOCAL_METADATA_DIR = LOCAL_OUTPUT_DIR / "metadata"

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/Cadence/Samples")

for directory in [LOCAL_WAV_DIR, LOCAL_MP3_DIR, LOCAL_METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Source recordings: {RECORDINGS_DIR}")
print(f"Local output: {LOCAL_OUTPUT_DIR}")
print(f"Drive output: {DRIVE_OUTPUT_DIR}")

Source recordings: /content/drive/MyDrive/Class Files/A2 CS 2025/Recordings
Local output: /content/cadence_dataset_samples
Drive output: /content/drive/MyDrive/Cadence/Samples


## 2. Mount Google Drive

In [2]:
from google.colab import drive

drive.mount("/content/drive")

if not RECORDINGS_DIR.exists():
    raise FileNotFoundError(
        f"Recordings directory not found: {RECORDINGS_DIR}. Check the Drive path."
    )

Mounted at /content/drive


## 3. Helper functions

In [3]:
def run_command(cmd: list[str]) -> subprocess.CompletedProcess[str]:
    return subprocess.run(cmd, capture_output=True, text=True)


def ffprobe_json(path: Path) -> dict[str, Any]:
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-print_format",
        "json",
        "-show_format",
        "-show_streams",
        str(path),
    ]
    result = run_command(cmd)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-1000:])
    return json.loads(result.stdout)


def audio_info(path: Path) -> dict[str, Any]:
    data = ffprobe_json(path)
    audio_stream = next(
        (
            stream
            for stream in data.get("streams", [])
            if stream.get("codec_type") == "audio"
        ),
        None,
    )
    if audio_stream is None:
        raise RuntimeError("No audio stream found.")

    duration = float(
        audio_stream.get("duration") or data.get("format", {}).get("duration") or 0.0
    )
    sample_rate = int(audio_stream.get("sample_rate") or 0)
    channels = int(audio_stream.get("channels") or 0)

    return {
        "duration_s": duration,
        "sample_rate": sample_rate,
        "channels": channels,
        "size_mb": path.stat().st_size / (1024 * 1024),
    }


def safe_stem(value: str, max_length: int = 80) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._-")
    return cleaned[:max_length] or "recording"


def size_mb(path: Path) -> float | None:
    return path.stat().st_size / (1024 * 1024) if path.exists() else None


def format_seconds(value: float) -> str:
    return f"{value:.3f}"

## 4. Audit MP4 recordings

In [4]:
mp4_files = sorted(RECORDINGS_DIR.glob("*.mp4"))
print(f"Found {len(mp4_files)} MP4 files in {RECORDINGS_DIR}")

audit_rows: list[dict[str, Any]] = []

for path in mp4_files:
    try:
        info = audio_info(path)
        status = (
            "eligible"
            if info["duration_s"] >= MIN_WINDOW_SECONDS
            else "skipped_short_duration"
        )
        error = ""

    except Exception as exc:
        info = {
            "duration_s": 0.0,
            "sample_rate": 0,
            "channels": 0,
            "size_mb": size_mb(path) or 0.0,
        }
        status = "skipped_probe_failed"
        error = str(exc)

    audit_rows.append(
        {
            "source_filename": path.name,
            "source_path": str(path),
            "source_duration_s": info["duration_s"],
            "source_size_mb": info["size_mb"],
            "source_sample_rate": info["sample_rate"],
            "source_channels": info["channels"],
            "status": status,
            "error": error,
            "path": path,
        }
    )

eligible_rows = [row for row in audit_rows if row["status"] == "eligible"]
print(f"Eligible files: {len(eligible_rows)}")

print(
    f"Skipped short files: {sum(row['status'] == 'skipped_short_duration' for row in audit_rows)}"
)
print(
    f"Probe failures: {sum(row['status'] == 'skipped_probe_failed' for row in audit_rows)}"
)

if len(eligible_rows) < SAMPLE_COUNT:
    raise RuntimeError(
        f"Need {SAMPLE_COUNT} eligible MP4 files, found {len(eligible_rows)}."
    )

Found 81 MP4 files in /content/drive/MyDrive/Class Files/A2 CS 2025/Recordings
Eligible files: 81
Skipped short files: 0
Probe failures: 0


## 5. Generate reproducible random sample windows

In [5]:
rng = random.Random(RANDOM_SEED)
selected_rows = rng.sample(eligible_rows, SAMPLE_COUNT)
sample_plan: list[dict[str, Any]] = []

for index, row in enumerate(selected_rows, start=1):
    source_duration_s = float(row["source_duration_s"])
    max_window_s = min(MAX_WINDOW_SECONDS, source_duration_s)
    window_duration_s = rng.uniform(MIN_WINDOW_SECONDS, max_window_s)
    window_start_s = rng.uniform(0.0, source_duration_s - window_duration_s)
    window_end_s = window_start_s + window_duration_s

    sample_id = f"sample_{index:03d}_{safe_stem(Path(row['source_filename']).stem)}"

    wav_local_path = LOCAL_WAV_DIR / f"{sample_id}.wav"
    mp3_local_path = LOCAL_MP3_DIR / f"{sample_id}.mp3"

    sample_plan.append(
        {
            "sample_id": sample_id,
            "source_filename": row["source_filename"],
            "source_path": row["source_path"],
            "source_duration_s": source_duration_s,
            "window_start_s": window_start_s,
            "window_end_s": window_end_s,
            "window_duration_s": window_duration_s,
            "wav_local_path": str(wav_local_path),
            "mp3_local_path": str(mp3_local_path),
            "wav_drive_path": "",
            "mp3_drive_path": "",
            "wav_size_mb": None,
            "mp3_size_mb": None,
            "sample_rate": WAV_SAMPLE_RATE,
            "channels": 1,
            "status": "planned",
            "error": "",
            "_source_path_obj": row["path"],
            "_wav_local_path_obj": wav_local_path,
            "_mp3_local_path_obj": mp3_local_path,
        }
    )

for item in sample_plan:
    print(
        f"{item['sample_id']}: start={item['window_start_s'] / 60:.2f}min, "
        f"duration={item['window_duration_s'] / 60:.2f}min, source={item['source_filename']}"
    )

sample_001_26_20-09-2025_Practical_P.4_-_Part_1_-_Seasons_Calculator_Functions_Procedures: start=10.56min, duration=10.16min, source=26_20-09-2025_Practical_P.4 - Part 1 - Seasons Calculator + Functions & Procedures.mp4
sample_002_14_09-08-2025_Practical_P.3_-_Part_2_-_WHILE_FOR_Loops: start=65.25min, duration=11.16min, source=14_09-08-2025_Practical_P.3 - Part 2 - WHILE & FOR Loops.mp4
sample_003_46_29-11-2025_Practical_P.6_-_Part_1_-_File_Handling_in_Python: start=79.38min, duration=12.81min, source=46_29-11-2025_Practical_P.6 - Part 1 - File Handling in Python.mp4
sample_004_41_13-11-2025_Theory_15.2_-_Part_2_-_Boolean_Simplification_Exercises: start=52.69min, duration=13.51min, source=41_13-11-2025_Theory_15.2 - Part 2 - Boolean Simplification Exercises.mp4
sample_005_39_06-11-2025_Theory_15.2_-_Part_1_-_Logic_Gates_Boolean_Algebra_Laws: start=29.82min, duration=12.25min, source=39_06-11-2025_Theory_15.2 - Part 1 - Logic Gates & Boolean Algebra Laws.mp4
sample_006_29_02-10-2025_The

## 6. Extract WAV and MP3 samples

In [6]:
def extract_wav(
    source_path: Path, output_path: Path, start_s: float, duration_s: float
) -> None:
    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-y",
        "-ss",
        format_seconds(start_s),
        "-i",
        str(source_path),
        "-t",
        format_seconds(duration_s),
        "-vn",
        "-ac",
        "1",
        "-ar",
        str(WAV_SAMPLE_RATE),
        "-acodec",
        "pcm_s16le",
        str(output_path),
    ]
    result = run_command(cmd)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-1000:])


def extract_mp3(
    source_path: Path, output_path: Path, start_s: float, duration_s: float
) -> None:
    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-y",
        "-ss",
        format_seconds(start_s),
        "-i",
        str(source_path),
        "-t",
        format_seconds(duration_s),
        "-vn",
        "-ac",
        "1",
        "-ar",
        str(MP3_SAMPLE_RATE),
        "-acodec",
        "libmp3lame",
        "-b:a",
        MP3_BITRATE,
        str(output_path),
    ]
    result = run_command(cmd)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-1000:])


for item in sample_plan:
    try:
        extract_wav(
            item["_source_path_obj"],
            item["_wav_local_path_obj"],
            item["window_start_s"],
            item["window_duration_s"],
        )
        extract_mp3(
            item["_source_path_obj"],
            item["_mp3_local_path_obj"],
            item["window_start_s"],
            item["window_duration_s"],
        )

        wav_info = audio_info(item["_wav_local_path_obj"])
        if wav_info["sample_rate"] != WAV_SAMPLE_RATE or wav_info["channels"] != 1:
            raise RuntimeError(
                f"WAV contract failed: {wav_info['sample_rate']}Hz, {wav_info['channels']} channels"
            )

        item["wav_size_mb"] = size_mb(item["_wav_local_path_obj"])
        item["mp3_size_mb"] = size_mb(item["_mp3_local_path_obj"])
        item["status"] = "completed_local"

        print(f"OK {item['sample_id']}")

    except Exception as exc:
        item["status"] = "failed"
        item["error"] = str(exc)
        print(f"FAIL {item['sample_id']}: {exc}")

OK sample_001_26_20-09-2025_Practical_P.4_-_Part_1_-_Seasons_Calculator_Functions_Procedures
OK sample_002_14_09-08-2025_Practical_P.3_-_Part_2_-_WHILE_FOR_Loops
OK sample_003_46_29-11-2025_Practical_P.6_-_Part_1_-_File_Handling_in_Python
OK sample_004_41_13-11-2025_Theory_15.2_-_Part_2_-_Boolean_Simplification_Exercises
OK sample_005_39_06-11-2025_Theory_15.2_-_Part_1_-_Logic_Gates_Boolean_Algebra_Laws
OK sample_006_29_02-10-2025_Theory_14.1_-_Part_2_-_TCP-IP
OK sample_007_25_18-09-2025_Theory_13.3_-_Part_6_-_Floating_Point_Errors
OK sample_008_84_20-03-2026_Theory_16.3_-_Part_1_-_Memory_Management_Paged_Memory_Memory_Level
OK sample_009_22_06-09-2025_Practical_P.3_-_Part_4_-_Employee_Salary_Calculation_Cont._Average
OK sample_010_69_31-01-2026_Practical_P.8_-_Part_1_-_Triangle_Calculator_Scenario


## 7. Copy samples to Google Drive

In [7]:
drive_wav_dir = DRIVE_OUTPUT_DIR / "wav"
drive_mp3_dir = DRIVE_OUTPUT_DIR / "mp3"
drive_metadata_dir = DRIVE_OUTPUT_DIR / "metadata"

for directory in [drive_wav_dir, drive_mp3_dir, drive_metadata_dir]:
    directory.mkdir(parents=True, exist_ok=True)

for item in sample_plan:
    if item["status"] != "completed_local":
        continue

    wav_drive_path = drive_wav_dir / item["_wav_local_path_obj"].name
    mp3_drive_path = drive_mp3_dir / item["_mp3_local_path_obj"].name

    shutil.copy2(item["_wav_local_path_obj"], wav_drive_path)
    shutil.copy2(item["_mp3_local_path_obj"], mp3_drive_path)

    item["wav_drive_path"] = str(wav_drive_path)
    item["mp3_drive_path"] = str(mp3_drive_path)
    item["status"] = "completed_drive_copy"

print(f"Copied completed samples to {DRIVE_OUTPUT_DIR}")

Copied completed samples to /content/drive/MyDrive/Cadence/Samples


## 8. Export dataset manifest CSV

In [8]:
manifest_columns = [
    "sample_id",
    "source_filename",
    "source_path",
    "source_duration_s",
    "window_start_s",
    "window_end_s",
    "window_duration_s",
    "wav_local_path",
    "mp3_local_path",
    "wav_drive_path",
    "mp3_drive_path",
    "wav_size_mb",
    "mp3_size_mb",
    "sample_rate",
    "channels",
    "status",
    "error",
]

manifest_path = LOCAL_METADATA_DIR / "dataset_manifest.csv"

with manifest_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=manifest_columns)
    writer.writeheader()
    for item in sample_plan:
        writer.writerow({column: item.get(column, "") for column in manifest_columns})

if DRIVE_OUTPUT_DIR is not None:
    drive_manifest_path = DRIVE_OUTPUT_DIR / "metadata" / manifest_path.name
    shutil.copy2(manifest_path, drive_manifest_path)
    print(f"Drive manifest: {drive_manifest_path}")

print(f"Local manifest: {manifest_path}")
print(
    f"Completed samples: {sum(item['status'].startswith('completed') for item in sample_plan)}"
)
print(f"Failed samples: {sum(item['status'] == 'failed' for item in sample_plan)}")

Drive manifest: /content/drive/MyDrive/Cadence/Samples/metadata/dataset_manifest.csv
Local manifest: /content/cadence_dataset_samples/metadata/dataset_manifest.csv
Completed samples: 10
Failed samples: 0
